# Colab: LettuceDetect Baseline Evaluation on CiteBench

This notebook evaluates **LettuceDetect** as a hallucination-detection baseline on CiteBench,
producing CiteEval metrics (CE, CA, CR) that can be directly compared with our own pipeline's scores.

**Workflow:**
1. Option A: Set `LETTUCE_SOURCE_SYSTEM_EVAL` in the config cell to an existing oracle-mode pipeline system JSON.
2. Option B: Leave it empty and run Step 3b to generate it automatically.
3. This notebook converts the JSON -> LettuceDetect input -> runs inference -> produces a new system JSON.
4. CiteEval scores the LettuceDetect output — compare CE/CA/CR directly with our pipeline.

**Why oracle?** Both pipelines see the same gold passages and the same LLM-generated responses.
Only the citation-verification step differs: our verifier vs. LettuceDetect. True apples-to-apples.

RAGTruth baseline is kept as optional (set `RUN_RAGTRUTH=True`), disabled by default.

Outputs are stored under: `/content/drive/MyDrive/AIST-FYP-colab-evals/<timestamp>/`

In [ ]:
# 0) Setup API Keys (Secrets)
try:
    from google.colab import userdata
    import os

    secret_keys = ["OPENAI_API_KEY", "DEEPSEEK_API_KEY", "HUGGINGFACE_TOKEN"]
    loaded_any = False

    for key in secret_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f"Loaded secret: {key}")
                loaded_any = True
        except Exception:
            pass

    if not loaded_any:
        print("No secrets loaded. Add keys in Colab Secrets if needed.")
except ImportError:
    print("google.colab.userdata not found. If running locally, export API keys manually.")

In [ ]:
# 1) Mount Drive and resolve workspace
from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    print(f'Not running in Colab or Drive mount failed: {exc}')

REPO_DIR = Path('/content/AIST-FYP')
if not REPO_DIR.exists():
    print('Repository not found at /content/AIST-FYP. Clone it in the next cell.')
else:
    print(f'Repository found: {REPO_DIR}')

In [ ]:
# 2) Clone repository (if needed) and install dependencies
import subprocess
import shlex


def run_cmd(cmd, cwd=None, check=True, stream=True):
    if isinstance(cmd, list):
        printable = ' '.join(shlex.quote(str(p)) for p in cmd)
    else:
        printable = cmd

    print(f'>> {printable}')

    if stream:
        if isinstance(cmd, list):
            process = subprocess.Popen(
                cmd,
                cwd=str(cwd) if cwd else None,
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                bufsize=1,
            )
        else:
            process = subprocess.Popen(
                cmd,
                cwd=str(cwd) if cwd else None,
                text=True,
                shell=True,
                executable='/bin/bash',
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                bufsize=1,
            )

        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            out_lines.append(line)
        process.wait()

        proc = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout=''.join(out_lines),
            stderr=None,
        )
    else:
        proc = subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            text=True,
            capture_output=True,
            shell=isinstance(cmd, str),
            executable='/bin/bash' if isinstance(cmd, str) else None,
        )
        if proc.stdout:
            print(proc.stdout)
        if proc.stderr:
            print(proc.stderr)

    if proc.returncode != 0 and check:
        raise RuntimeError(f'Command failed ({proc.returncode}): {printable}')

    return proc


if not REPO_DIR.exists():
    # Update this URL if your fork differs
    run_cmd(['git', 'clone', 'https://github.com/xiashuidaolaoshuren/AIST-FYP.git', str(REPO_DIR)])

os.chdir(REPO_DIR)

# Preferred: uv project sync for colab env
uv_ok = run_cmd('which uv', check=False).returncode == 0
if not uv_ok:
    run_cmd('curl -LsSf https://astral.sh/uv/install.sh | sh', check=False)
    os.environ['PATH'] = f"{Path.home() / '.local' / 'bin'}:{os.environ.get('PATH','')}"

sync_proc = run_cmd(['uv', 'sync', '--project', 'colab/env', '--extra', 'evaluation'], check=False)
if sync_proc.returncode != 0:
    run_cmd(['pip', 'install', '-r', 'requirements.txt'])

# Force a stable NumPy/Transformers stack on Colab.
# This addresses intermittent broken numpy binary states (e.g., ImportError from numpy._core.umath).
run_cmd([
    'pip', 'install', '--upgrade',
    'numpy',
    'transformers',
    'tokenizers',
    'huggingface_hub',
])

# Ensure runtime dependencies for this workflow, including CiteEval extras used at eval time.
run_cmd(['pip', 'install', '-U', 'lettucedetect', 'python-dotenv', 'configobj', 'faiss-cpu', 'rank_bm25'])

# Provision NLTK resources required by CiteEval conversion.
run_cmd([
    'python', '-c',
    'import nltk; '
    'nltk.download("punkt", quiet=True); '
    'nltk.download("punkt_tab", quiet=True); '
    'print("nltk punkt resources ready")'
])

# Quick sanity check so failures happen early in setup, not deep in pipeline cells.
run_cmd([
    'python', '-c',
    'import numpy, transformers, configobj, nltk; '
    'from nltk.data import find; '
    'find("tokenizers/punkt_tab/english/"); '
    'print("numpy", numpy.__version__); '
    'print("transformers", transformers.__version__); '
    'print("configobj", configobj.__version__); '
    'print("nltk punkt_tab ok")'
])

# Test for numpy import issues (common in Colab if binary state is broken)
try:
    from numpy._core.umath import _center
    print('NumPy import test passed.')
except ImportError as e:
    print(f'NumPy import test failed: {e}')

print('Dependency setup complete.')
print('If import errors persist in an existing session, use Runtime -> Restart runtime, then re-run from the top.')

### 4. Configuration (Smoke Test Settings)
By default, this notebook runs a **smoke test** with a limited number of samples (`MAX_SAMPLES = 10`). 

**To run the full evaluation:**
1. Verify that the smoke test completes successfully.
2. Change `MAX_SAMPLES = 10` to `MAX_SAMPLES = None` in the cell below.
3. Re-run the notebook.

In [ ]:
# 3) Runtime configuration (edit this cell only)
from datetime import datetime
from pathlib import Path
import os
import shutil

# Data and scale
METRIC_SPLIT = 'test'  # dev or test
MAX_SAMPLES = 10       # smoke default
STRICT = True
INCLUDE_FLAT_CONTEXT = True

# Drive artifact locations
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/data')
DRIVE_CITEEVAL_ROOT = Path('/content/drive/MyDrive/AIST-FYP/benchmark/CiteEval')

# Evaluation provider defaults
CITEEVAL_PROVIDER = 'deepseek'
EVAL_MODEL_NAME = 'deepseek-chat'
MODULES = 'ca,ce,cr_itercoe,cr_editdist'
VERSION = 'citeeval-auto-12272024'
CONTEXT_SOURCE = 'oracle'

# LettuceDetect model and downstream citation mode
LETTUCE_MODEL_PATH = 'KRLabsOrg/lettucedect-base-modernbert-en-v1'
LETTUCE_USE_SPAN_CITATIONS = True
LETTUCE_CONFIDENCE_THRESHOLD = 0.5

# CiteEval system JSON produced by our oracle-mode pipeline run.
# (output of: evaluate_mitigation_citebench.py --context-source oracle)
# Schema: [{id, query, passages (oracle), pred}, ...]
LETTUCE_SOURCE_SYSTEM_EVAL = ''

# The same verifier-output system JSON used as the direct "our verifier" evaluation input.
# Leave empty to auto-fill from cell 3b, or set manually to an existing system_eval_full_verifier.json.
PIPELINE_VERIFIER_SYSTEM_EVAL = ''

# Pipeline oracle run settings (used by cell 3b to generate LETTUCE_SOURCE_SYSTEM_EVAL)
# Use full_verifier for fair comparison: same generator prose, only citation verification differs.
PIPELINE_VARIANT = 'full_verifier'         # verifier variant to run
PIPELINE_ORACLE_DATASET = 'asqa'           # 'asqa', 'eli5', or 'msmarco'
PIPELINE_OUTPUT_DIR = ''                   # leave empty for auto timestamped output under outputs/
PIPELINE_RESUME = False                    # set True to resume an incomplete run
PIPELINE_STRATEGY = 'validation'           # retrieval artifact strategy to load in pipeline runtime
PIPELINE_CONFIG_PATH = 'config.colab.yaml' # auto-created in cell 3b if missing

# RAGTruth mode
# - 'inline_inference': convert CiteBench -> RAGTruth JSONL, then run pretrained baseline inference, then adapt to CiteEval format
# - 'convert_only': convert CiteBench -> RAGTruth-style JSONL, then adapt to CiteEval format
# - 'prediction_input': use an existing RAGTruth prediction file and convert it
RAGTRUTH_MODE = 'inline_inference'
RAGTRUTH_PREDICTION_INPUT = ''  # e.g. '/content/drive/MyDrive/path/to/prediction.jsonl'
RAGTRUTH_TRAIN_OUTPUT_ROOT = Path('/content/drive/MyDrive/AIST-FYP-colab-outputs/ragtruth_baseline/train_outputs')
RAGTRUTH_CHECKPOINT_OVERRIDE = ''  # optional exact checkpoint folder path
RAGTRUTH_MAX_NEW_TOKENS = 256

# RAGTruth baseline — disabled by default (set True only if you want RAGTruth comparison)
RUN_RAGTRUTH = False

# Resume control: set to an existing run directory to continue in-place
RESUME_RUN_DIR = ''


def _ensure_symlink_dir(link_path: Path, target_path: Path) -> None:
    target_path = Path(target_path)
    link_path = Path(link_path)
    target_path.mkdir(parents=True, exist_ok=True)
    link_path.parent.mkdir(parents=True, exist_ok=True)

    if link_path.is_symlink():
        current_target = Path(os.readlink(link_path))
        if current_target == target_path:
            return
        link_path.unlink()
    elif link_path.exists():
        if link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()

    os.symlink(target_path, link_path, target_is_directory=True)


# Link CiteEval benchmark from Drive into repo when available
project_citeeval_root = REPO_DIR / 'benchmark' / 'CiteEval'
if DRIVE_CITEEVAL_ROOT.exists():
    _ensure_symlink_dir(project_citeeval_root, DRIVE_CITEEVAL_ROOT)
    print(f'CiteEval benchmark path: {project_citeeval_root} -> {project_citeeval_root.resolve()}')
else:
    print(
        f'Warning: DRIVE_CITEEVAL_ROOT not found at {DRIVE_CITEEVAL_ROOT}. '
        'If benchmark is stored elsewhere on Drive, update DRIVE_CITEEVAL_ROOT.'
    )

SOURCE_METRIC_FILE = REPO_DIR / 'benchmark' / 'CiteEval' / 'data' / 'metric_eval' / f'metric_{METRIC_SPLIT}' / f'citebench.metric_{METRIC_SPLIT}'

# Drive output root
DRIVE_ROOT = Path('/content/drive/MyDrive/AIST-FYP-colab-evals')
if RESUME_RUN_DIR:
    RUN_DIR = Path(RESUME_RUN_DIR)
    if not RUN_DIR.exists():
        raise FileNotFoundError(f'RESUME_RUN_DIR does not exist: {RUN_DIR}')
    print(f'Resuming existing run dir: {RUN_DIR}')
else:
    RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
    RUN_DIR = DRIVE_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Created run dir: {RUN_DIR}')

RAG_DIR = RUN_DIR / 'ragtruth'
LETTUCE_DIR = RUN_DIR / 'lettucedetect'
EVAL_DIR = RUN_DIR / 'evaluation'
for p in [RAG_DIR, LETTUCE_DIR, EVAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('SOURCE_METRIC_FILE =', SOURCE_METRIC_FILE)
print('RUN_DIR =', RUN_DIR)
print('RAGTRUTH_MODE =', RAGTRUTH_MODE)
print('LETTUCE_USE_SPAN_CITATIONS =', LETTUCE_USE_SPAN_CITATIONS)
print('LETTUCE_CONFIDENCE_THRESHOLD =', LETTUCE_CONFIDENCE_THRESHOLD)
print('LETTUCE_SOURCE_SYSTEM_EVAL =', LETTUCE_SOURCE_SYSTEM_EVAL if LETTUCE_SOURCE_SYSTEM_EVAL else '<NOT SET>')
print('PIPELINE_VERIFIER_SYSTEM_EVAL =', PIPELINE_VERIFIER_SYSTEM_EVAL if PIPELINE_VERIFIER_SYSTEM_EVAL else '<NOT SET>')
print('PIPELINE_VARIANT =', PIPELINE_VARIANT)
print('PIPELINE_ORACLE_DATASET =', PIPELINE_ORACLE_DATASET)
print('PIPELINE_STRATEGY =', PIPELINE_STRATEGY)
print('PIPELINE_CONFIG_PATH =', PIPELINE_CONFIG_PATH)
print('RUN_RAGTRUTH =', RUN_RAGTRUTH)

In [ ]:
# 3b) Generate LETTUCE_SOURCE_SYSTEM_EVAL via our oracle-mode pipeline (skip if already set)
#
# This cell runs evaluate_mitigation_citebench.py --context-source oracle to produce
# the CiteEval system JSON that LettuceDetect will process in Step 6.
#
# For fair comparison, set PIPELINE_VARIANT='full_verifier' so both methods see the same
# generator prose and oracle passages, differing only in citation-verification strategy.
#
# If you already have a system_eval JSON from a previous run, set LETTUCE_SOURCE_SYSTEM_EVAL
# and PIPELINE_VERIFIER_SYSTEM_EVAL in the config cell above and skip this cell.

from pathlib import Path
import sys
import yaml


if LETTUCE_SOURCE_SYSTEM_EVAL and PIPELINE_VERIFIER_SYSTEM_EVAL:
    print(f'LETTUCE_SOURCE_SYSTEM_EVAL already set: {LETTUCE_SOURCE_SYSTEM_EVAL}')
    print(f'PIPELINE_VERIFIER_SYSTEM_EVAL already set: {PIPELINE_VERIFIER_SYSTEM_EVAL}')
    print('Skipping pipeline run. Clear values in config to re-generate.')
else:
    # Ensure colab-tuned config exists (mirrors verifier notebook behavior).
    pipeline_config_path = REPO_DIR / PIPELINE_CONFIG_PATH
    if not pipeline_config_path.exists():
        base_config = REPO_DIR / 'config.yaml'
        if not base_config.exists():
            raise FileNotFoundError(f'Base config not found: {base_config}')

        with open(base_config, 'r', encoding='utf-8') as f:
            cfg = yaml.safe_load(f)

        cfg.setdefault('processing', {})['device'] = 'cuda'
        cfg.setdefault('verification', {}).setdefault('nli', {})['device'] = 'cuda'
        cfg['verification'].setdefault('self_agreement', {})['device'] = 'cuda'
        cfg.setdefault('retrieval', {}).setdefault('faiss', {})['use_gpu'] = False
        cfg['retrieval']['faiss']['gpu_id'] = 0

        with open(pipeline_config_path, 'w', encoding='utf-8') as f:
            yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
        print(f'Wrote pipeline config: {pipeline_config_path}')

    # Ensure required retrieval artifacts exist for pipeline initialization.
    # The script initializes BaselineRAGPipeline, which validates FAISS/metadata/chunks paths
    # even for oracle context mode.
    faiss_index = REPO_DIR / f'data/indexes/{PIPELINE_STRATEGY}/faiss.index'
    index_meta = REPO_DIR / f'data/indexes/{PIPELINE_STRATEGY}/metadata.pkl'
    chunks_file = REPO_DIR / f'data/processed/wiki_chunks_{PIPELINE_STRATEGY}.jsonl'

    if not (faiss_index.exists() and index_meta.exists() and chunks_file.exists()) and DRIVE_DATA_ROOT.exists():
        print('Required retrieval artifacts not found under repo data/. Attempting Drive data linkage...')
        _ensure_symlink_dir(REPO_DIR / 'data', DRIVE_DATA_ROOT)
        faiss_index = REPO_DIR / f'data/indexes/{PIPELINE_STRATEGY}/faiss.index'
        index_meta = REPO_DIR / f'data/indexes/{PIPELINE_STRATEGY}/metadata.pkl'
        chunks_file = REPO_DIR / f'data/processed/wiki_chunks_{PIPELINE_STRATEGY}.jsonl'

    missing = [
        p for p in [faiss_index, index_meta, chunks_file]
        if not p.exists()
    ]
    if missing:
        raise FileNotFoundError(
            'Missing retrieval artifacts required by evaluate_mitigation_citebench.py runtime init:\n'
            + '\n'.join(f'- {p}' for p in missing)
            + '\n\nFix options:\n'
            + f'1) Ensure DRIVE_DATA_ROOT exists and points to your artifacts: {DRIVE_DATA_ROOT}\n'
            + f'2) Change PIPELINE_STRATEGY to the strategy you actually have (current: {PIPELINE_STRATEGY})\n'
            + '3) Build artifacts in repo: process_wikipedia.py, generate_embeddings.py, build_faiss_index.py'
        )

    python_exec = sys.executable
    pipeline_cmd = [
        python_exec, 'scripts/evaluate_mitigation_citebench.py',
        '--config', PIPELINE_CONFIG_PATH,
        '--dataset-role', 'mitigation',
        '--strategy', PIPELINE_STRATEGY,
        '--context-source', 'oracle',
        '--oracle-dataset', PIPELINE_ORACLE_DATASET,
        '--variants', PIPELINE_VARIANT,
        '--provider', CITEEVAL_PROVIDER,
        '--model-name', EVAL_MODEL_NAME,
        '--modules', MODULES,
        '--version', VERSION,
    ]
    if MAX_SAMPLES is not None:
        pipeline_cmd += ['--max-samples', str(MAX_SAMPLES)]
    if PIPELINE_OUTPUT_DIR:
        pipeline_cmd += ['--output-dir', PIPELINE_OUTPUT_DIR]
    if PIPELINE_RESUME:
        pipeline_cmd += ['--resume']

    run_cmd(pipeline_cmd, cwd=REPO_DIR)

    # Resolve generated system_eval JSON path
    if PIPELINE_OUTPUT_DIR:
        pipeline_root = Path(PIPELINE_OUTPUT_DIR) / 'oracle' / 'system_inputs'
    else:
        # Auto-discover latest run under outputs/mitigation_eval_citebench
        base = REPO_DIR / 'outputs' / 'mitigation_eval_citebench'
        runs = sorted(base.glob('*/oracle/system_inputs'), key=lambda p: p.stat().st_mtime, reverse=True)
        if not runs:
            raise FileNotFoundError(f'No oracle system_inputs found under {base}')
        pipeline_root = runs[0]

    generated_path = pipeline_root / f'system_eval_{PIPELINE_VARIANT}.json'
    if not generated_path.exists():
        raise FileNotFoundError(
            f'Expected system_eval JSON not found: {generated_path}\n'
            'Check that evaluate_mitigation_citebench.py completed without errors.'
        )

    LETTUCE_SOURCE_SYSTEM_EVAL = str(generated_path)
    PIPELINE_VERIFIER_SYSTEM_EVAL = str(generated_path)
    print(f'Generated LETTUCE_SOURCE_SYSTEM_EVAL: {LETTUCE_SOURCE_SYSTEM_EVAL}')
    print(f'Generated PIPELINE_VERIFIER_SYSTEM_EVAL: {PIPELINE_VERIFIER_SYSTEM_EVAL}')

In [ ]:
# 4) Load API keys and evaluator env
import json
import os

if IN_COLAB:
    try:
        from google.colab import userdata
        if CITEEVAL_PROVIDER == 'deepseek':
            os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
            os.environ.setdefault('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
        else:
            os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception as exc:
        print(f'Could not load secret from Colab userdata: {exc}')

os.environ['CITEEVAL_PROVIDER'] = CITEEVAL_PROVIDER
os.environ['CITEEVAL_ROOT'] = str(REPO_DIR / 'benchmark' / 'CiteEval')
extra_pythonpath = os.pathsep.join([
    str(REPO_DIR / 'benchmark' / 'CiteEval'),
    str(REPO_DIR / 'benchmark' / 'CiteEval' / 'src'),
])
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH','')}{os.pathsep if os.environ.get('PYTHONPATH') else ''}{extra_pythonpath}"

if CITEEVAL_PROVIDER == 'deepseek' and not os.environ.get('DEEPSEEK_API_KEY'):
    raise RuntimeError('DEEPSEEK_API_KEY is required for DeepSeek evaluation.')
if CITEEVAL_PROVIDER == 'openai' and not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY is required for OpenAI evaluation.')

print('Evaluator provider:', CITEEVAL_PROVIDER)
print('Environment ready.')

In [ ]:
# 5) [RAGTruth - optional, controlled by RUN_RAGTRUTH flag]
ragtruth_system_eval = RAG_DIR / 'ragtruth_system_eval.json'  # placeholder path
if RUN_RAGTRUTH:
    import json
    import os
    import re
    from pathlib import Path

    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer

    ragtruth_metric_jsonl = RAG_DIR / f'ragtruth_metric_{METRIC_SPLIT}.jsonl'
    ragtruth_aligned_ids = RAG_DIR / 'aligned_ids.json'
    ragtruth_convert_report = RAG_DIR / 'convert_metric_report.json'
    ragtruth_system_eval = RAG_DIR / 'ragtruth_system_eval.json'
    ragtruth_system_report = RAG_DIR / 'ragtruth_to_citeeval_report.json'
    ragtruth_prediction_output = RAG_DIR / 'ragtruth_predictions.inline.jsonl'


    def _load_jsonl_rows(path: Path) -> list[dict]:
        rows: list[dict] = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                payload = line.strip()
                if not payload:
                    continue
                row = json.loads(payload)
                if isinstance(row, dict):
                    rows.append(row)
        return rows


    def _resolve_ragtruth_checkpoint(train_root: Path, override: str | None) -> Path:
        if override:
            override_path = Path(override)
            if not override_path.exists():
                raise FileNotFoundError(f'RAGTRUTH_CHECKPOINT_OVERRIDE does not exist: {override_path}')
            return override_path

        if not train_root.exists():
            raise FileNotFoundError(
                f'RAGTruth train output root not found: {train_root}. '
                'Set RAGTRUTH_CHECKPOINT_OVERRIDE or train baseline first.'
            )

        candidate_dirs: list[Path] = []
        for marker in ('config.json', 'adapter_config.json'):
            for p in train_root.glob(f'**/{marker}'):
                candidate_dirs.append(p.parent)

        if not candidate_dirs:
            raise FileNotFoundError(
                f'No checkpoint directories with config.json/adapter_config.json found under: {train_root}'
            )

        unique_dirs = list({str(p.resolve()): p.resolve() for p in candidate_dirs}.values())
        return max(unique_dirs, key=lambda p: os.path.getmtime(p))


    def _parse_prediction(text: str) -> dict:
        text = text.strip()

        try:
            pred = json.loads(text)
            if isinstance(pred, dict) and 'hallucination list' in pred:
                return pred
        except Exception:
            pass

        for match in re.finditer(r'\{[\s\S]*?\}', text):
            try:
                pred = json.loads(match.group(0))
                if isinstance(pred, dict) and 'hallucination list' in pred:
                    return pred
            except Exception:
                continue

        list_match = re.search(r'hallucination list[^\[]*(\[[\s\S]*?\])', text, flags=re.I)
        if list_match:
            try:
                hall_list = json.loads(list_match.group(1))
                if isinstance(hall_list, list):
                    return {'hallucination list': hall_list}
            except Exception:
                pass

        return {'hallucination list': []}


    def _build_prompt(sample: dict) -> str:
        question = str(sample.get('question', '')).strip()
        reference = str(sample.get('reference', '')).strip()
        response = str(sample.get('response', '')).strip()
        return (
            'Below is a question:\n'
            f'{question}\n\n'
            'Below are related passages:\n'
            f'{reference}\n\n'
            'Below is an answer:\n'
            f'{response}\n\n'
            'Your task is to determine whether the summary contains either or both of the following two types of hallucinations:\n'
            '1. conflict: instances where the summary presents direct contraction or opposition to the original news;\n'
            '2. baseless info: instances where the generated summary includes information which is not substantiated by or inferred from the original news.\n'
            'Then, compile the labeled hallucinated spans into a JSON dict, with a key "hallucination list" and its value is a list of hallucinated spans. '
            'If there exist potential hallucinations, the output should be in the following JSON format: '
            '{"hallucination list": [hallucination span1, hallucination span2, ...]}. '
            'Otherwise, leave the value as an empty list as following: {"hallucination list": []}.\n'
            'Output only valid JSON with key "hallucination list" and no extra text:'
        )


    def _build_generation_input(tokenizer, prompt_text: str) -> str:
        if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
            messages = [
                {
                    'role': 'system',
                    'content': (
                        'You are a strict JSON generator. Reply with only a JSON object with key "hallucination list". '
                        'No explanations, no markdown, no reasoning.'
                    ),
                },
                {'role': 'user', 'content': prompt_text},
            ]
            try:
                return tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                return tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
        return f'[INST] {prompt_text.strip()} [/INST]'


    ragtruth_input_for_adapter: Path
    if RAGTRUTH_MODE == 'prediction_input':
        if not RAGTRUTH_PREDICTION_INPUT:
            raise ValueError('RAGTRUTH_PREDICTION_INPUT must be set when RAGTRUTH_MODE=prediction_input')
        ragtruth_input_for_adapter = Path(RAGTRUTH_PREDICTION_INPUT)
    else:
        run_cmd([
            'python', 'scripts/convert_citebench_metric_to_ragtruth.py',
            '--input', str(SOURCE_METRIC_FILE),
            '--output', str(ragtruth_metric_jsonl),
            '--split', METRIC_SPLIT,
            '--aligned-ids-output', str(ragtruth_aligned_ids),
            '--report-json', str(ragtruth_convert_report),
        ] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else []) + (['--strict'] if STRICT else []), cwd=REPO_DIR)

        if RAGTRUTH_MODE == 'inline_inference':
            rows = _load_jsonl_rows(ragtruth_metric_jsonl)
            target_ids = {str(row.get('id', '')) for row in rows}
            print(f'RAGTruth inline inference rows: {len(rows)}')

            existing_rows: list[dict] = []
            existing_ids: set[str] = set()
            if ragtruth_prediction_output.exists():
                for row in _load_jsonl_rows(ragtruth_prediction_output):
                    sample_id = str(row.get('id', ''))
                    if not sample_id:
                        raise ValueError(f'Resume mismatch: missing id in {ragtruth_prediction_output}')
                    if sample_id not in target_ids:
                        raise ValueError(
                            f'Resume mismatch: existing prediction id {sample_id} not in current dataset selection.'
                        )
                    if sample_id in existing_ids:
                        raise ValueError(
                            f'Resume mismatch: duplicate prediction id {sample_id} in {ragtruth_prediction_output}'
                        )
                    existing_ids.add(sample_id)
                    existing_rows.append(row)
                if len(existing_rows) > len(rows):
                    raise ValueError(
                        f'Resume mismatch: existing predictions ({len(existing_rows)}) exceed target rows ({len(rows)}).'
                    )
                print(f'Resume progress: {len(existing_rows)} complete, {len(rows) - len(existing_rows)} remaining')

            rows_to_run = [row for row in rows if str(row.get('id', '')) not in existing_ids]

            checkpoint_path = _resolve_ragtruth_checkpoint(
                train_root=RAGTRUTH_TRAIN_OUTPUT_ROOT,
                override=RAGTRUTH_CHECKPOINT_OVERRIDE if RAGTRUTH_CHECKPOINT_OVERRIDE else None,
            )
            print('Resolved RAGTruth checkpoint:', checkpoint_path)

            tokenizer = AutoTokenizer.from_pretrained(str(checkpoint_path), trust_remote_code=True)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                str(checkpoint_path),
                torch_dtype='auto',
                device_map='auto',
                trust_remote_code=True,
            )
            model.eval()

            write_mode = 'a' if existing_rows else 'w'
            with open(ragtruth_prediction_output, write_mode, encoding='utf-8') as f:
                for row in tqdm(rows_to_run):
                    prompt_text = _build_prompt(row)
                    gen_input_text = _build_generation_input(tokenizer, prompt_text)

                    inputs = tokenizer(gen_input_text, return_tensors='pt').to(model.device)
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=RAGTRUTH_MAX_NEW_TOKENS,
                        do_sample=False,
                        pad_token_id=tokenizer.pad_token_id,
                    )

                    gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
                    pred = _parse_prediction(gen_text)

                    enriched = dict(row)
                    enriched['generated_response'] = json.dumps(pred, ensure_ascii=False)
                    enriched['pred'] = pred
                    enriched['raw_pred_text'] = gen_text
                    f.write(json.dumps(enriched, ensure_ascii=False) + '\n')

            ragtruth_input_for_adapter = ragtruth_prediction_output
            print('RAGTruth inline prediction file:', ragtruth_prediction_output)
        else:
            ragtruth_input_for_adapter = ragtruth_metric_jsonl

    run_cmd([
        'python', 'scripts/convert_ragtruth_baseline_to_citeeval.py',
        '--input', str(ragtruth_input_for_adapter),
        '--output', str(ragtruth_system_eval),
        '--report-json', str(ragtruth_system_report),
    ] + (['--strict'] if STRICT else []), cwd=REPO_DIR)

    if ragtruth_system_report.exists():
        with open(ragtruth_system_report, 'r', encoding='utf-8') as f:
            report = json.load(f)
        total_output = int(report.get('total_output', 0) or 0)
        if total_output == 0:
            raise RuntimeError(
                'RAGTruth conversion produced 0 system-eval rows. '
                f'Report: {ragtruth_system_report}. '
                'If this happens in inline mode, ensure your Colab repo includes the latest '
                'scripts/convert_ragtruth_baseline_to_citeeval.py fix that preserves input passages.'
            )

    print('RAGTruth adapter input:', ragtruth_input_for_adapter)
    print('RAGTruth system-eval file:', ragtruth_system_eval)
else:
    print('RAGTruth step skipped (RUN_RAGTRUTH=False). Set RUN_RAGTRUTH=True in config to enable.')

In [ ]:
# 6) Run LettuceDetect oracle-track pipeline (convert precomputed system input + inference + output conversion)
import json
from pathlib import Path

if not LETTUCE_SOURCE_SYSTEM_EVAL:
    raise ValueError(
        'LETTUCE_SOURCE_SYSTEM_EVAL is not set.\n'
        'Point it to the CiteEval system JSON produced by our own RAG pipeline '
        '(e.g. outputs/full_pipeline_queries_....json or a Drive path) that contains '
        'oracle passages (id, query, passages, pred fields).'
    )

lettuce_source_system_eval = Path(LETTUCE_SOURCE_SYSTEM_EVAL)
if not lettuce_source_system_eval.exists():
    raise FileNotFoundError(f'Missing source CiteEval system file: {lettuce_source_system_eval}')

lettuce_preconverted_input = LETTUCE_DIR / 'lettucedetect_input.oracle.json'
run_cmd([
    'python', 'scripts/convert_citebench_oracle_to_lettucedetect.py',
    '--input', str(lettuce_source_system_eval),
    '--output', str(lettuce_preconverted_input),
] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else [])
  + (['--strict'] if STRICT else [])
  + (['--include-flat-context'] if INCLUDE_FLAT_CONTEXT else []), cwd=REPO_DIR)

run_cmd([
    'python', 'scripts/run_lettucedetect_pipeline.py',
    '--preconverted-input', str(lettuce_preconverted_input),
    '--model-path', LETTUCE_MODEL_PATH,
    '--output-dir', str(LETTUCE_DIR),
] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else [])
  + (['--strict'] if STRICT else [])
  + (['--include-flat-context'] if INCLUDE_FLAT_CONTEXT else [])
  + (['--use-span-citations'] if LETTUCE_USE_SPAN_CITATIONS else [])
  + (['--confidence-threshold', str(LETTUCE_CONFIDENCE_THRESHOLD)] if LETTUCE_USE_SPAN_CITATIONS else []), cwd=REPO_DIR)

lettuce_manifest = LETTUCE_DIR / 'run_manifest.json'
if not lettuce_manifest.exists():
    raise FileNotFoundError(f'Missing LettuceDetect manifest: {lettuce_manifest}')

with open(lettuce_manifest, 'r', encoding='utf-8') as f:
    manifest_obj = json.load(f)

stats = manifest_obj.get('inference_stats', {})
print('LettuceDetect stats:', stats)
if int(stats.get('errors', 0)) > 0:
    raise RuntimeError(f'LettuceDetect inference reported errors: {stats}')

lettuce_system_eval = Path(manifest_obj['system_eval_output'])
print('LettuceDetect source system file:', lettuce_source_system_eval)
print('LettuceDetect preconverted input:', lettuce_preconverted_input)
print('LettuceDetect system-eval file:', lettuce_system_eval)
print('LettuceDetect citation mode:', 'span_injected' if LETTUCE_USE_SPAN_CITATIONS else 'original')

In [ ]:
# 7a) [RAGTruth - optional, controlled by RUN_RAGTRUTH flag]
if RUN_RAGTRUTH:
    import json

    RAG_EVAL_DIR = EVAL_DIR / 'ragtruth_eval'
    RAG_EVAL_DIR.mkdir(parents=True, exist_ok=True)

    run_cmd([
        'python', 'scripts/evaluate_citebench_method.py',
        '--method-name', 'ragtruth_baseline',
        '--system-input', str(ragtruth_system_eval),
        '--provider', CITEEVAL_PROVIDER,
        '--model-name', EVAL_MODEL_NAME,
        '--modules', MODULES,
        '--version', VERSION,
        '--context-source', CONTEXT_SOURCE,
        '--output-dir', str(RAG_EVAL_DIR),
    ] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else []), cwd=REPO_DIR)

    ragtruth_summary_path = RAG_EVAL_DIR / 'summary.json'
    if not ragtruth_summary_path.exists():
        raise FileNotFoundError(f'Missing RAGTruth summary: {ragtruth_summary_path}')

    with open(ragtruth_summary_path, 'r', encoding='utf-8') as f:
        ragtruth_summary = json.load(f)

    ragtruth_returncode = ragtruth_summary.get('command_returncode')
    if ragtruth_returncode != 0:
        raise RuntimeError(f'RAGTruth evaluation failed with command_returncode={ragtruth_returncode}')

    ragtruth_rows = ragtruth_summary.get('run', {}).get('evaluated_rows')
    print('RAGTruth evaluated rows =', ragtruth_rows)
    if not isinstance(ragtruth_rows, int) or ragtruth_rows <= 0:
        raise RuntimeError(f'RAGTruth evaluation produced no rows: evaluated_rows={ragtruth_rows}')
else:
    print('RAGTruth step skipped (RUN_RAGTRUTH=False). Set RUN_RAGTRUTH=True in config to enable.')

In [ ]:
# 7b) Evaluate LettuceDetect with CiteEval settings
import json

LETTUCE_EVAL_DIR = EVAL_DIR / 'lettucedetect_eval'
LETTUCE_EVAL_DIR.mkdir(parents=True, exist_ok=True)

run_cmd([
    'python', 'scripts/evaluate_citebench_method.py',
    '--method-name', 'lettucedetect',
    '--system-input', str(lettuce_system_eval),
    '--provider', CITEEVAL_PROVIDER,
    '--model-name', EVAL_MODEL_NAME,
    '--modules', MODULES,
    '--version', VERSION,
    '--context-source', CONTEXT_SOURCE,
    '--output-dir', str(LETTUCE_EVAL_DIR),
] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else []), cwd=REPO_DIR)

lettuce_summary_path = LETTUCE_EVAL_DIR / 'summary.json'
if not lettuce_summary_path.exists():
    raise FileNotFoundError(f'Missing LettuceDetect summary: {lettuce_summary_path}')

with open(lettuce_summary_path, 'r', encoding='utf-8') as f:
    lettuce_summary = json.load(f)

lettuce_returncode = lettuce_summary.get('command_returncode')
if lettuce_returncode != 0:
    raise RuntimeError(f'LettuceDetect evaluation failed with command_returncode={lettuce_returncode}')

lettuce_rows = lettuce_summary.get('run', {}).get('evaluated_rows')
print('LettuceDetect evaluated rows =', lettuce_rows)
if not isinstance(lettuce_rows, int) or lettuce_rows <= 0:
    raise RuntimeError(f'LettuceDetect evaluation produced no rows: evaluated_rows={lettuce_rows}')

In [ ]:
# 8a) [RAGTruth - optional, controlled by RUN_RAGTRUTH flag]
if RUN_RAGTRUTH:
    from datetime import datetime, timezone
    import json

    ragtruth_eval_dir = EVAL_DIR / 'ragtruth_eval'
    ragtruth_summary_path = ragtruth_eval_dir / 'summary.json'
    if not ragtruth_summary_path.exists():
        raise FileNotFoundError(
            f'Missing RAGTruth summary: {ragtruth_summary_path}. '
            'Run the RAGTruth evaluation command first.'
        )

    with open(ragtruth_summary_path, 'r', encoding='utf-8') as f:
        ragtruth_summary = json.load(f)

    ragtruth_metrics = {
        'method': 'ragtruth_baseline',
        'split': METRIC_SPLIT,
        'sample_count': ragtruth_summary.get('run', {}).get('evaluated_rows'),
        'provider': CITEEVAL_PROVIDER,
        'model_name': EVAL_MODEL_NAME,
        'modules': MODULES,
        'metrics': ragtruth_summary.get('module_metrics', {}),
        'artifact_paths': {
            'system_eval_input': str(ragtruth_system_eval),
            'summary': str(ragtruth_summary_path),
        },
        'timestamp': datetime.now(timezone.utc).isoformat(),
    }

    ragtruth_metrics_path = EVAL_DIR / 'ragtruth_metrics.json'
    with open(ragtruth_metrics_path, 'w', encoding='utf-8') as f:
        json.dump(ragtruth_metrics, f, indent=2, ensure_ascii=False)

    ragtruth_manifest = {
        'run_dir': str(RUN_DIR),
        'metric_source': str(SOURCE_METRIC_FILE),
        'ragtruth_mode': RAGTRUTH_MODE,
        'provider': CITEEVAL_PROVIDER,
        'model_name': EVAL_MODEL_NAME,
        'max_samples': MAX_SAMPLES,
        'strict': STRICT,
        'method': 'ragtruth_baseline',
        'paths': {
            'ragtruth_metrics': str(ragtruth_metrics_path),
            'ragtruth_summary': str(ragtruth_summary_path),
        },
    }

    ragtruth_manifest_path = RUN_DIR / 'run_manifest_ragtruth.json'
    with open(ragtruth_manifest_path, 'w', encoding='utf-8') as f:
        json.dump(ragtruth_manifest, f, indent=2, ensure_ascii=False)

    print('Saved RAGTruth JSON artifacts:')
    print('-', ragtruth_metrics_path)
    print('-', ragtruth_manifest_path)
else:
    print('RAGTruth step skipped (RUN_RAGTRUTH=False). Set RUN_RAGTRUTH=True in config to enable.')

In [ ]:
# 8b) Save LettuceDetect canonical evaluation JSON artifact on Drive
from datetime import datetime, timezone
import json

lettuce_eval_dir = EVAL_DIR / 'lettucedetect_eval'
lettuce_summary_path = lettuce_eval_dir / 'summary.json'
if not lettuce_summary_path.exists():
    raise FileNotFoundError(
        f'Missing LettuceDetect summary: {lettuce_summary_path}. '
        'Run the LettuceDetect evaluation command first.'
    )

with open(lettuce_summary_path, 'r', encoding='utf-8') as f:
    lettuce_summary = json.load(f)

lettuce_metrics = {
    'method': 'lettucedetect',
    'split': METRIC_SPLIT,
    'sample_count': lettuce_summary.get('run', {}).get('evaluated_rows'),
    'provider': CITEEVAL_PROVIDER,
    'model_name': EVAL_MODEL_NAME,
    'modules': MODULES,
    'metrics': lettuce_summary.get('module_metrics', {}),
    'artifact_paths': {
        'system_eval_input': str(lettuce_system_eval),
        'summary': str(lettuce_summary_path),
    },
    'timestamp': datetime.now(timezone.utc).isoformat(),
}

lettuce_metrics_path = EVAL_DIR / 'lettucedetect_metrics.json'
with open(lettuce_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(lettuce_metrics, f, indent=2, ensure_ascii=False)

lettuce_manifest = {
    'run_dir': str(RUN_DIR),
    'metric_source': str(SOURCE_METRIC_FILE),
    'ragtruth_mode': RAGTRUTH_MODE,
    'provider': CITEEVAL_PROVIDER,
    'model_name': EVAL_MODEL_NAME,
    'max_samples': MAX_SAMPLES,
    'strict': STRICT,
    'method': 'lettucedetect',
    'paths': {
        'lettucedetect_metrics': str(lettuce_metrics_path),
        'lettucedetect_summary': str(lettuce_summary_path),
    },
}

lettuce_manifest_path = RUN_DIR / 'run_manifest_lettucedetect.json'
with open(lettuce_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(lettuce_manifest, f, indent=2, ensure_ascii=False)

print('Saved LettuceDetect JSON artifacts:')
print('-', lettuce_metrics_path)
print('-', lettuce_manifest_path)

## Notes

- `RAGTRUTH_MODE='inline_inference'` runs pretrained RAGTruth baseline inference on converted CiteBench rows.
- Inline mode resolves checkpoint in this order: `RAGTRUTH_CHECKPOINT_OVERRIDE` first, otherwise newest valid folder under `RAGTRUTH_TRAIN_OUTPUT_ROOT`.
- `RAGTRUTH_MODE='convert_only'` uses CiteBench predictions as the response field for a baseline-compatible conversion path.
- `RAGTRUTH_MODE='prediction_input'` converts an existing baseline prediction file from `RAGTRUTH_PREDICTION_INPUT`.
- Setup cell now bootstraps CiteEval runtime dependencies including NLTK sentence-tokenizer resources (`punkt`, `punkt_tab`).
- Evaluation is split into two independent cells:
  - `7a` runs `ragtruth_baseline` evaluation and writes `evaluation/ragtruth_eval/summary.json`.
  - `7b` runs `lettucedetect` evaluation and writes `evaluation/lettucedetect_eval/summary.json`.
- Metrics export is split into two independent save cells:
  - `8a` writes `evaluation/ragtruth_metrics.json` and `run_manifest_ragtruth.json`.
  - `8b` writes `evaluation/lettucedetect_metrics.json` and `run_manifest_lettucedetect.json`.
- You can run either evaluation/save path independently after its corresponding system-eval input exists.